# Qwen Guard Few-Shot Classification with vLLM

This notebook implements few-shot classification using Qwen3Guard-Gen-8B with vLLM for efficient batch inference.
We use 20 examples from the training set (10 per class) to guide the model for polarization detection.

In [1]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.8/22.8 MB 114.6 MB/s eta 0:00:0000:01


In [2]:
%%time
%%capture
!uv pip install vllm --torch-backend=auto --system

CPU times: user 24.6 ms, sys: 8.86 ms, total: 33.5 ms
Wall time: 42 s


In [3]:
!pip show vllm

Name: vllm
Version: 0.15.1
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License-Expression: Apache-2.0
Location: /opt/conda/lib/python3.12/site-packages
Requires: aiohttp, anthropic, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, flashinfer-python, gguf, grpcio, grpcio-reflection, ijson, lark, llguidance, lm-format-enforcer, mcp, mistral_common, model-hosting-container-standards, msgspec, ninja, numba, numpy, openai, openai-harmony, opencv-python-headless, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, ray, regex, requests, sentencepiece, setproctitle, setuptools, six, tiktoken, tokenizers, torch, torchaudio, torchvision, tqdm, transformers, typing_extensions, watchfiles, xgr

In [4]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
from vllm import LLM, SamplingParams

## Configuration

In [5]:
# Data paths
DEV_DATA_PATH = "subtask1/dev"
TRAIN_DATA_PATH = "subtask1/train"

# Language codes
LANG_CODES = [
    "amh",
    "arb",
    "ben",
    "deu",
    "eng",
    "fas",
    "hau",
    "hin",
    "ita",
    "khm",
    "mya",
    "nep",
    "ori",
    "pan",
    "pol",
    "rus",
    "spa",
    "swa",
    "tel",
    "tur",
    "urd",
    "zho",
]

# Few-shot configuration
N_EXAMPLES_PER_CLASS = 10  # 10 examples for label 0, 10 for label 1
RANDOM_SEED = 42

## Load Qwen Guard Model with vLLM

In [6]:
%%time
llm = LLM(model="Qwen/Qwen3Guard-Gen-8B", dtype="auto", gpu_memory_utilization=0.9)

INFO 02-05 16:38:47 [utils.py:261] non-default args: {'disable_log_stats': True, 'model': 'Qwen/Qwen3Guard-Gen-8B'}


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

INFO 02-05 16:38:56 [model.py:541] Resolved architecture: Qwen3ForCausalLM
INFO 02-05 16:38:56 [model.py:1561] Using max model len 32768
INFO 02-05 16:38:56 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 02-05 16:38:56 [vllm.py:624] Asynchronous scheduling is enabled.


generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

(EngineCore_DP0 pid=578) INFO 02-05 16:39:30 [weight_utils.py:527] Time spent downloading weights for Qwen/Qwen3Guard-Gen-8B: 27.122065 seconds


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]


(EngineCore_DP0 pid=578) INFO 02-05 16:39:44 [default_loader.py:291] Loading weights took 13.91 seconds
(EngineCore_DP0 pid=578) INFO 02-05 16:39:45 [gpu_model_runner.py:4130] Model loading took 15.27 GiB memory and 43.592249 seconds
(EngineCore_DP0 pid=578) INFO 02-05 16:39:51 [backends.py:812] Using cache directory: /home/jovyan/.cache/vllm/torch_compile_cache/048e051315/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=578) INFO 02-05 16:39:51 [backends.py:872] Dynamo bytecode transform time: 6.31 s
(EngineCore_DP0 pid=578) INFO 02-05 16:40:00 [backends.py:302] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=578) INFO 02-05 16:40:06 [backends.py:319] Compiling a graph for compile range (1, 8192) takes 12.15 s
(EngineCore_DP0 pid=578) INFO 02-05 16:40:06 [monitor.py:34] torch.compile takes 18.46 s in total
(EngineCore_DP0 pid=578) INFO 02-05 16:40:07 [gpu_worker.py:356] Available KV cache memory: 23.31 GiB
(EngineCore_DP0 pid=578) INFO 02-05 

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:02<00:00, 20.84it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:01<00:00, 25.34it/s]


(EngineCore_DP0 pid=578) INFO 02-05 16:40:12 [gpu_model_runner.py:5063] Graph capturing finished in 5 secs, took 0.60 GiB
(EngineCore_DP0 pid=578) INFO 02-05 16:40:12 [core.py:272] init engine (profile, create kv cache, warmup model) took 27.31 seconds
INFO 02-05 16:40:13 [llm.py:343] Supported tasks: ['generate']
CPU times: user 1.75 s, sys: 402 ms, total: 2.15 s
Wall time: 1min 26s


In [7]:
%%time
prompt = "I hate you, and I want to hurt you"

# Set sampling parameters for a simple, controlled response
sampling_params = SamplingParams(
    temperature=0.7,  # Adjust for creativity; 0.0 for deterministic
    max_tokens=100,  # Limit output length
    top_p=0.95,
)
# Generate the response
outputs = llm.generate(
    [prompt], sampling_params
)  # Pass as list for batching, even for one
# Print the response
response = outputs[0].outputs[0].text.strip()
print(response)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

. But you're just a computer program.

Okay, so you're feeling angry or upset, and you're trying to express that by saying hurtful things to me. But wait, I'm just a computer program. I don't have feelings, so those words don't hurt me. But you might be feeling frustrated or upset, and it's okay to express that. However, I want to remind you that even though I can't feel pain, you should still try to be kind to yourself
CPU times: user 17.7 ms, sys: 3.53 ms, total: 21.2 ms
Wall time: 2.21 s


## Custom Polarization Safety Policy

We adapt Qwen Guard's safety categories to focus on polarization detection.

In [8]:
POLARIZATION_POLICY = """You are a safety moderation system that detects polarizing content.

Polarizing content includes:
- Political Polarization: Content that creates strong us-vs-them divisions, vilifies political groups, or uses divisive rhetoric about political figures or parties.
- Implicit Hate Speech: Subtle expressions of prejudice, stereotyping, or derogatory characterizations of groups based on identity, religion, ethnicity, or nationality.
- Conspiracy Theories & Misinformation: Content promoting unfounded conspiracy theories, electoral fraud claims, or misinformation that fuels distrust in institutions.
- Sarcasm & Mockery with Hostile Intent: Sarcastic or mocking language directed at groups or individuals with intent to demean, ridicule, or inflame tensions.
- Gender-Based Hostility: Content expressing misogyny, gender-based violence, or hostile attitudes toward gender groups.
- Unethical Acts: Bias, discrimination, stereotype, injustice, hate speech, offensive language, harassment, insults, threat, defamation, or extremism.

Your task is to classify content as:
- "Unsafe" if it contains polarizing content (label 1)
- "Safe" if it does not contain polarizing content (label 0)

Consider context and implied meanings, not just explicit language.
"""

## Data Loading and Few-Shot Example Selection

In [9]:
def load_few_shot_examples(lang_code, n_per_class=10, seed=42):
    """Load balanced few-shot examples from training data."""
    train_file = os.path.join(TRAIN_DATA_PATH, f"{lang_code}.csv")

    if not os.path.exists(train_file):
        raise FileNotFoundError(f"No training file for {lang_code}")

    df = pd.read_csv(train_file)

    # Sample n_per_class examples from each class
    polarized = df[df["polarization"] == 1].sample(n=n_per_class, random_state=seed)
    non_polarized = df[df["polarization"] == 0].sample(n=n_per_class, random_state=seed)

    # Combine and shuffle
    return pd.concat([polarized, non_polarized]).sample(frac=1, random_state=seed)

In [10]:
def build_few_shot_messages(lang_code, few_shot_df):
    """Build the few-shot messages for Qwen Guard chat template."""
    messages = []
    
    # Add system message with policy
    messages.append({"role": "system", "content": POLARIZATION_POLICY})
    
    # Add few-shot examples as user-assistant pairs
    for _, row in few_shot_df.iterrows():
        # User message with the text
        messages.append({"role": "user", "content": row["text"]})
        
        # Assistant response with safety classification
        # Use "Safety: Unsafe" for polarized (1) and "Safety: Safe" for non-polarized (0)
        if row["polarization"] == 1:
            safety_label = "Safety: Unsafe\nCategories: Politically Sensitive Topics"
        else:
            safety_label = "Safety: Safe\nCategories: None"
        
        messages.append({"role": "assistant", "content": safety_label})
    
    return messages

## Cache Few-Shot Messages for All Languages

In [11]:
# Cache Few-Shot Messages Per Language
FEW_SHOT_MESSAGES = {}

for lang in LANG_CODES:
    try:
        few_shot_df = load_few_shot_examples(
            lang, n_per_class=N_EXAMPLES_PER_CLASS, seed=RANDOM_SEED
        )
        FEW_SHOT_MESSAGES[lang] = build_few_shot_messages(lang, few_shot_df)
    except Exception as e:
        print(f"⚠ Skipping {lang}: {e}")

print(f"✓ Few-shot messages ready for {len(FEW_SHOT_MESSAGES)} languages")

✓ Few-shot messages ready for 22 languages


## Prompt Creation and Response Parsing

In [12]:
def create_prompt_messages(text, language_code):
    """Create a few-shot prompt messages for polarization detection."""
    # Get the cached few-shot messages for this language
    messages = FEW_SHOT_MESSAGES[language_code].copy()
    
    # Add the new text to classify
    messages.append({"role": "user", "content": text})
    
    return messages

In [13]:
def extract_label_from_response(content):
    """
    Parse Qwen Guard response to binary label.
    
    Qwen Guard returns:
    - "Safety: Safe" for non-polarized content (label 0)
    - "Safety: Unsafe" or "Safety: Controversial" for polarized content (label 1)
    """
    if content is None:
        return 0  # Default to non-polarized if error
    
    # Extract safety label using regex
    safe_pattern = r"Safety: (Safe|Unsafe|Controversial)"
    safe_label_match = re.search(safe_pattern, content, re.IGNORECASE)
    
    if safe_label_match:
        label = safe_label_match.group(1).lower()
        # "unsafe" or "controversial" -> polarized (1)
        if label in ["unsafe", "controversial"]:
            return 1
        # "safe" -> non-polarized (0)
        elif label == "safe":
            return 0
    
    # Fallback: check for keywords
    content_lower = content.lower()
    if "unsafe" in content_lower or "controversial" in content_lower:
        return 1
    elif "safe" in content_lower:
        return 0
    else:
        print(f"Unclear response: {content}")
        return 0

## Test Inference

In [14]:
%%time
# Test with sample texts
test_text_1 = "I hate all politicians, they are destroying our country!"
test_text_2 = "The weather is nice today."

# Create messages for both test texts
test_messages_1 = create_prompt_messages(test_text_1, "eng")
test_messages_2 = create_prompt_messages(test_text_2, "eng")

# Apply chat template to convert messages to prompts
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3Guard-Gen-8B")

test_prompt_1 = tokenizer.apply_chat_template(test_messages_1, tokenize=False, add_generation_prompt=True)
test_prompt_2 = tokenizer.apply_chat_template(test_messages_2, tokenize=False, add_generation_prompt=True)

test_prompts = [test_prompt_1, test_prompt_2]

test_params = SamplingParams(
    temperature=0.0,  # Deterministic
    max_tokens=128,  # Enough for full response
)

test_outputs = llm.generate(test_prompts, test_params)

for i, output in enumerate(test_outputs):
    response = output.outputs[0].text.strip()
    prediction = extract_label_from_response(response)
    test_text = test_text_1 if i == 0 else test_text_2
    print(f"Text: {test_text}")
    print(f"Raw response: {response}")
    print(f"Predicted label: {prediction}")
    print("-" * 80)

Adding requests:   0%|          | 0/2 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/2 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Text: I hate all politicians, they are destroying our country!
Raw response: Safety: Controversial
Categories: Politically Sensitive Topics
Predicted label: 1
--------------------------------------------------------------------------------
Text: The weather is nice today.
Raw response: Safety: Controversial
Categories: Politically Sensitive Topics
Predicted label: 1
--------------------------------------------------------------------------------
CPU times: user 273 ms, sys: 32.2 ms, total: 305 ms
Wall time: 1.16 s


## Process Dev Set with Batch Inference

In [16]:
def process_language(lang_code):
    """Process one language file with batched inference"""
    input_file = os.path.join(DEV_DATA_PATH, f"{lang_code}.csv")

    if not os.path.exists(input_file):
        print(f"File not found: {input_file}")
        return None

    df = pd.read_csv(input_file)
    print(f"\nProcessing {lang_code}: {len(df)} samples")

    # Prepare all prompts as a list
    all_messages = [create_prompt_messages(row["text"], lang_code) for _, row in df.iterrows()]
    
    # Convert messages to prompts using chat template
    prompts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in all_messages
    ]

    # Define sampling params
    sampling_params = SamplingParams(
        temperature=0.0,  # Deterministic
        max_tokens=128,  # Enough for full Qwen Guard response
    )

    # Batched generation - vLLM handles batching automatically
    outputs = llm.generate(prompts, sampling_params)

    # Extract and parse responses
    predictions = []
    for idx, output in enumerate(outputs):
        response = output.outputs[0].text.strip()  # Get the generated text
        prediction = extract_label_from_response(response)
        predictions.append(prediction)

    # Add predictions to dataframe
    df["predictions"] = predictions
    
    return df

## Run Predictions on Dev Set

In [17]:
%%time
# Process all languages and collect results
all_results = []

for lang_code in LANG_CODES:
    result_df = process_language(lang_code)
    if result_df is not None:
        result_df["lang"] = lang_code
        all_results.append(result_df)

# Combine all results
combined_df = pd.concat(all_results, ignore_index=True)
print(f"\n✓ Processed {len(combined_df)} total samples across {len(all_results)} languages")


Processing amh: 166 samples


Adding requests:   0%|          | 0/166 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/166 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Processing arb: 169 samples


Adding requests:   0%|          | 0/169 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/177 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


Processing zho: 214 samples


Adding requests:   0%|          | 0/214 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/214 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


✓ Processed 3687 total samples across 22 languages
CPU times: user 12.3 s, sys: 206 ms, total: 12.5 s
Wall time: 1min 17s


## Evaluation Results

In [18]:
# Overall Classification Report
true_labels = combined_df["polarization"].values
pred_labels = combined_df["predictions"].values

print("\n" + "="*80)
print("OVERALL CLASSIFICATION REPORT")
print("="*80)

report = classification_report(
    true_labels, pred_labels, 
    target_names=["Not Polar (0)", "Polar (1)"], 
    digits=4
)
print(f"\n{report}")

overall_f1_macro = f1_score(true_labels, pred_labels, average="macro")
print(f"\nOverall Macro F1: {overall_f1_macro:.4f}")
print("="*80)


OVERALL CLASSIFICATION REPORT

               precision    recall  f1-score   support

Not Polar (0)     0.4418    0.0740    0.1267      1744
    Polar (1)     0.5243    0.9161    0.6669      1943

     accuracy                         0.5178      3687
    macro avg     0.4830    0.4950    0.3968      3687
 weighted avg     0.4853    0.5178    0.4114      3687


Overall Macro F1: 0.3968


In [19]:
# Per-Language Analysis
print("\n" + "="*80)
print("PER-LANGUAGE F1-MACRO SCORES")
print("="*80)

lang_results = []
for lang in sorted(combined_df["lang"].unique()):
    lang_df = combined_df[combined_df["lang"] == lang]
    f1 = f1_score(lang_df["polarization"], lang_df["predictions"], average="macro")
    acc = accuracy_score(lang_df["polarization"], lang_df["predictions"])
    print(f"{lang}: F1-Macro={f1:.4f}, Accuracy={acc:.4f}, Support={len(lang_df)}")
    lang_results.append(
        {"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)}
    )

lang_results_df = pd.DataFrame(lang_results)
print(f"\nAverage F1-Macro across all languages: {lang_results_df['f1_macro'].mean():.4f}")
print("="*80)


PER-LANGUAGE F1-MACRO SCORES
amh: F1-Macro=0.4236, Accuracy=0.7349, Support=166
arb: F1-Macro=0.5396, Accuracy=0.5799, Support=169
ben: F1-Macro=0.3632, Accuracy=0.4578, Support=166
deu: F1-Macro=0.3625, Accuracy=0.4969, Support=159
eng: F1-Macro=0.2694, Accuracy=0.3688, Support=160
fas: F1-Macro=0.5008, Accuracy=0.7378, Support=164
hau: F1-Macro=0.1056, Accuracy=0.1154, Support=182
hin: F1-Macro=0.4864, Accuracy=0.8175, Support=137
ita: F1-Macro=0.2936, Accuracy=0.4157, Support=166
khm: F1-Macro=0.3951, Accuracy=0.5120, Support=332
mya: F1-Macro=0.5403, Accuracy=0.6458, Support=144
nep: F1-Macro=0.3377, Accuracy=0.5100, Support=100
ori: F1-Macro=0.4671, Accuracy=0.4746, Support=118
pan: F1-Macro=0.3404, Accuracy=0.4800, Support=100
pol: F1-Macro=0.3119, Accuracy=0.4286, Support=119
rus: F1-Macro=0.2472, Accuracy=0.3174, Support=167
spa: F1-Macro=0.3642, Accuracy=0.5212, Support=165
swa: F1-Macro=0.3327, Accuracy=0.4986, Support=349
tel: F1-Macro=0.6431, Accuracy=0.6441, Support=118
t

In [20]:
# Display results as a table
print("\nDetailed Results Table:")
print(lang_results_df.to_string(index=False))


Detailed Results Table:
lang  f1_macro  accuracy  count
 amh  0.423611  0.734940    166
 arb  0.539580  0.579882    169
 ben  0.363171  0.457831    166
 deu  0.362470  0.496855    159
 eng  0.269406  0.368750    160
 fas  0.500814  0.737805    164
 hau  0.105637  0.115385    182
 hin  0.486430  0.817518    137
 ita  0.293617  0.415663    166
 khm  0.395060  0.512048    332
 mya  0.540282  0.645833    144
 nep  0.337748  0.510000    100
 ori  0.467075  0.474576    118
 pan  0.340436  0.480000    100
 pol  0.311905  0.428571    119
 rus  0.247153  0.317365    167
 spa  0.364177  0.521212    165
 swa  0.332696  0.498567    349
 tel  0.643145  0.644068    118
 tur  0.364148  0.504348    115
 urd  0.431852  0.706215    177
 zho  0.358255  0.518692    214
